## 2 CodeAgent Example: Broker Agent

In [ ]:
import yfinance as yf
from smolagents import tool

# A simple wallet class to manage stocks and balance
class myWallet:
    def __init__(self, balance: float):
        self.balance = balance
        self.stocks = {}

    def add_stock(self, ticker: str, quantity: int, stock_price: float):
        if self.balance < quantity * stock_price:
            return {"status": "error", "message": "Insufficient balance to buy stocks."}
        
        self.balance -= quantity * stock_price
        if ticker in self.stocks:
            self.stocks[ticker] += quantity
        else:
            self.stocks[ticker] = quantity
        return {"status": "success", "balance": self.balance, "stocks": self.stocks}
    
    def remove_stock(self, ticker: str, quantity: int, stock_price: float):
        if ticker not in self.stocks or self.stocks[ticker] < quantity:
            return {"status": "error", "message": f"Not enough shares of {ticker} to sell."}
        
        self.balance += quantity * stock_price
        self.stocks[ticker] -= quantity
        if self.stocks[ticker] == 0:
            del self.stocks[ticker]
        return {"status": "success", "balance": self.balance, "stocks": self.stocks}
    


# Helper function to fetch stock price using yfinance
def get_stock_price(ticker: str) -> float:
    """
    Fetches the current stock price for the given ticker symbol.

    Args:
        ticker: Stock ticker symbol (e.g., "AAPL", "GOOGL").

    Returns:
        The current stock price as a float.
    """
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        price = info.get("regularMarketPrice")
        return price
    except Exception as e:
        print(f"Could not fetch stock price for {ticker}: {e}")
        return -1
    

wallet = myWallet(10000)  # Initialize wallet with $10,000

#### Excercise 1:

Implement a new tool called `sell_stocks` that allows the agent to sell a specified quantity of stocks for a given ticker symbol. The tool should check if the agent owns enough shares to sell and update the wallet balance accordingly.

In [ ]:
@tool
def sell_stocks(ticker: str, quantity: int) -> str:
    """
    Sells a specified quantity of stocks for the given ticker symbol.
    
    Args:
        ticker: Stock ticker symbol (e.g., "AAPL", "GOOGL").
        quantity: Number of shares to sell.
    Returns:
        A string confirming the sale or an error message.    
    """
    price = get_stock_price(ticker)
    
    if price > -1:
        response = wallet.remove_stock(ticker, quantity, price)
        if response["status"] == "success":
            return f"Sold {quantity} shares of {ticker} at ${price:.2f} each. New balance: ${response['balance']:.2f}"
        else:
            return response["message"]
    else:
        return f"Could not fetch stock price for {ticker}."


#### Exercise 2:

The agent currently supports four tools: `list_files`, `read_file`, `write_file`,
and `delete_file`. Your task is to extend it with two additional tools and then
test the full system using an **interactive conversation loop**.

**Implement the following tools:**

- **`rename_file(old_name, new_name)`** — renames a file inside the sandbox using
  `os.rename`. It should return a clear error message if the source file does not exist.
- **`count_words(filename)`** — reads a file and returns the number of words it contains,
  for example: `"File 'notes.txt' contains 14 words."`. Handle the case where
  the file does not exist.

Once the tools are implemented, rebuild the agent with all six tools and run
the **interactive loop** in the next cell. The loop maintains a conversation history
so the agent remembers what happened in previous turns — you can refer to a file
created earlier without re-explaining the context.


In [ ]:
@tool
def rename_file(old_name: str, new_name: str) -> str:
    """
    Renames a file in the sandbox.
    Args:
        old_name: The current name of the file.
        new_name: The new name for the file.
    Returns:
        A success or error message.
    """
    import os
    try:
        os.rename(old_name, new_name)
        return f"File '{old_name}' has been renamed to '{new_name}' successfully."
    except FileNotFoundError:
        return f"Error: File '{old_name}' not found."
    except Exception as e:
        return f"Error renaming file '{old_name}': {e}"


@tool
def count_words(filename: str) -> str:
    """
    Counts the number of words in a file.
    Args:
        filename: The name of the file to read.
    Returns:
        The word count as a string, or an error message.
    """
    try:
        with open(filename, "r") as f:
            content = f.read()
        word_count = len(content.split())
        return f"The file '{filename}' contains {word_count} words."
    except FileNotFoundError:
        return f"Error: File '{filename}' not found."
    except Exception as e:
        return f"Error counting words in file '{filename}': {e}"
